# Evaluate BLEU and COMET for Whisper Predictions

## models:
- pred_before: whisper-small
- pred_after: whisper-small-yue (cantonese), whisper-small-az (azerbeijani)

## details about finetune:
- whisper-small-yue: trained with fleurs parallel data (cantonese and english),  trained using language code = chinese
- whisper-small-az: trained with fleurs parallel data (azerbeijani and english), trained using language code = turkish

## prediction output:
- cantonese: predicted using language code = chinese
- azerbeijani: predicted using language code = azerbeijani


In [1]:
# # Install from source
# !pip install git+https://github.com/Unbabel/COMET.git


In [3]:
!pip install sacrebleu
!pip install unbabel-comet
# !pip install unbabel-comet>=2.0.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 155.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 179.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 137.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 210.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 187.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 191.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 215.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 221.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 221.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 262.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 132.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.

In [1]:
import importlib.metadata

print(importlib.metadata.version("unbabel-comet"))

2.2.7


In [2]:
import pandas as pd
from comet import download_model, load_from_checkpoint
from sacrebleu import corpus_bleu
from sacrebleu.metrics import BLEU

In [3]:
import comet
print(comet.__file__)
print(comet.__version__ if hasattr(comet, "__version__") else "No version")

/opt/conda/lib/python3.12/site-packages/comet/__init__.py
2.2.7


In [4]:
# -----------------------
# Load data
# -----------------------

df = pd.read_csv("your_file.tsv", sep="\t")

# -----------------------
# COMET
# -----------------------

model_path = download_model("Unbabel/wmt22-comet-da")
model = load_from_checkpoint(model_path)

before_data = [
    {
        "src": row.text_source,
        "mt": row.pred_before,
        "ref": row.text_en,
    }
    for row in df.itertuples(index=False)
]

after_data = [
    {
        "src": row.text_source,
        "mt": row.pred_after,
        "ref": row.text_en,
    }
    for row in df.itertuples(index=False)
]

before_output = model.predict(
    before_data,
    batch_size=16,
    gpus=1,      # change to 0 if using CPU
)

after_output = model.predict(
    after_data,
    batch_size=16,
    gpus=1,
)

# sentence-level COMET
df["comet_before"] = before_output.scores
df["comet_after"] = after_output.scores

# corpus-level COMET
corpus_comet_before = before_output.system_score
corpus_comet_after = after_output.system_score

# -----------------------
# SacreBLEU
# -----------------------

bleu = BLEU(effective_order=True)

# sentence-level BLEU
df["bleu_before"] = [
    bleu.sentence_score(pred, [ref]).score
    for pred, ref in zip(df.pred_before, df.text_en)
]

df["bleu_after"] = [
    bleu.sentence_score(pred, [ref]).score
    for pred, ref in zip(df.pred_after, df.text_en)
]

# corpus-level BLEU
corpus_bleu_before = corpus_bleu(
    df.pred_before.tolist(),
    [df.text_en.tolist()],
).score

corpus_bleu_after = corpus_bleu(
    df.pred_after.tolist(),
    [df.text_en.tolist()],
).score


# -----------------------
# CHRF
# -----------------------

# from sacrebleu.metrics import CHRF

# chrf = CHRF()

# corpus_chrf_before = chrf.corpus_score(
#     df.pred_before.tolist(),
#     [df.text_en.tolist()],
# ).score

# corpus_chrf_after = chrf.corpus_score(
#     df.pred_after.tolist(),
#     [df.text_en.tolist()],
# ).score

# df["chrf_before"] = [
#     chrf.sentence_score(pred, [ref]).score
#     for pred, ref in zip(df.pred_before, df.text_en)
# ]

# df["chrf_after"] = [
#     chrf.sentence_score(pred, [ref]).score
#     for pred, ref in zip(df.pred_after, df.text_en)
# ]


# -----------------------
# Winner
# -----------------------

def compare(before, after):
    if after > before:
        return "after"
    elif before > after:
        return "before"
    else:
        return "tie"

df["winner_comet"] = [
    compare(b, a)
    for b, a in zip(df.comet_before, df.comet_after)
]

df["winner_bleu"] = [
    compare(b, a)
    for b, a in zip(df.bleu_before, df.bleu_after)
]

# -----------------------
# Save
# -----------------------

df.to_csv(
    "evaluated_w_scores.tsv",
    sep="\t",
    index=False,
)

# -----------------------
# Summary
# -----------------------

print("=" * 60)
print("CORPUS RESULTS")
print("=" * 60)

print(f"COMET Before : {corpus_comet_before:.4f}")
print(f"COMET After  : {corpus_comet_after:.4f}")

print()

print(f"SacreBLEU Before : {corpus_bleu_before:.2f}")
print(f"SacreBLEU After  : {corpus_bleu_after:.2f}")

print()

print("Sentence winners (COMET)")
print(df["winner_comet"].value_counts())

print()

print("Sentence winners (BLEU)")
print(df["winner_bleu"].value_counts())

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

LICENSE: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

Encoder model frozen.
/opt/conda/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA L40S') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/gener

CORPUS RESULTS
COMET Before : 0.5889
COMET After  : 0.6208

SacreBLEU Before : 4.10
SacreBLEU After  : 5.17

Sentence winners (COMET)
winner_comet
after     568
before    355
Name: count, dtype: int64

Sentence winners (BLEU)
winner_bleu
after     495
before    427
tie         1
Name: count, dtype: int64
